# 🧪 Bloque 4 — Sesión 2: ANOVA, Mutual Information y documentación para el modelado

**Módulo:** Depuración, Limpieza y Clasificación de Datos (5109)  
**Unidad Didáctica:** UD2 — Verificación estadística de la calidad de los datos  
**Duración estimada:** ~4 horas

---

### Contenidos de esta sesión
- **4.3** ANOVA de un factor: comparación de medias entre más de dos grupos y Tukey
- **4.4** Información mutua (Mutual Information): dependencia no lineal
- **4.5** Documentación de resultados y toma de decisiones para el modelado

### Criterios de evaluación asociados
> **(CE a)** Se han descrito las técnicas estadísticas de análisis de la calidad técnica de los datos.  
> **(CE b)** Se han aplicado técnicas de evaluación basadas en cálculos estadísticos para detectar la cobertura y el sesgo.  
> **(CE c)** Se ha documentado la descripción del resultado de las verificaciones.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import f_oneway, kruskal, shapiro, levene
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from sklearn.feature_selection import mutual_info_classif, mutual_info_regression
from scipy.stats import pearsonr

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
np.random.seed(42)

titanic = sns.load_dataset('titanic')
diamonds = sns.load_dataset('diamonds')
penguins = sns.load_dataset('penguins').dropna()

print('✅ Librerías y datasets cargados')

---
## 4.3 ANOVA de un factor

### ¿Qué pregunta responde?

Cuando queremos comparar medias entre **más de dos grupos**, aplicar t-tests en pares múltiples infla el error de tipo I. El **ANOVA** compara simultáneamente todos los grupos:

- **H₀:** μ₁ = μ₂ = … = μₖ
- **H₁:** al menos un grupo tiene media distinta

$$F = \frac{MS_{between}}{MS_{within}}$$

### Condiciones
1. Los grupos son independientes
2. Variable dependiente numérica continua
3. Distribución aproximadamente normal en cada grupo
4. Varianzas similares entre grupos (homocedasticidad)

In [ ]:
# ANOVA: ¿difiere el precio según el tipo de corte?
grupos_cut = [diamonds.loc[diamonds['cut'] == c, 'price'].values
              for c in diamonds['cut'].unique()]

F, p = f_oneway(*grupos_cut)

print('Estadísticos descriptivos por tipo de corte:')
resumen = diamonds.groupby('cut')['price'].agg(['mean', 'median', 'std', 'count'])
resumen.columns = ['Media', 'Mediana', 'Std', 'n']
print(resumen.round(2))
print()
print(f'ANOVA de un factor: F={F:.4f}, p={p:.6f}')
if p < 0.05:
    print('✅ Rechazamos H₀: al menos un tipo de corte tiene precio medio distinto')
else:
    print('❌ No rechazamos H₀')

In [ ]:
orden_cut = ['Fair', 'Good', 'Very Good', 'Premium', 'Ideal']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=diamonds, x='cut', y='price', order=orden_cut,
            palette='Blues', ax=axes[0],
            flierprops=dict(marker='o', markersize=2, alpha=0.3))
axes[0].set_title(f'Precio por tipo de corte\nF={F:.2f}, p={p:.2e}')
axes[0].set_xlabel('Corte')
axes[0].set_ylabel('Precio ($)')
axes[0].tick_params(axis='x', rotation=15)

medias = diamonds.groupby('cut')['price'].mean().reindex(orden_cut)
sems = diamonds.groupby('cut')['price'].sem().reindex(orden_cut)
axes[1].bar(orden_cut, medias, color='steelblue', edgecolor='white',
            yerr=1.96*sems, capsize=5, alpha=0.8)
axes[1].set_title('Media ± IC95% por tipo de corte')
axes[1].set_xlabel('Corte')
axes[1].set_ylabel('Precio medio ($)')
axes[1].tick_params(axis='x', rotation=15)

plt.suptitle('ANOVA — Precio de diamantes por tipo de corte', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Prueba post-hoc de Tukey HSD

ANOVA solo nos dice que **al menos un grupo difiere**. Para identificar los pares concretos se usa la prueba **Tukey HSD**, que compara todos los pares controlando el error acumulado.

In [ ]:
df_anova = diamonds[['cut', 'price']].dropna()
tukey = pairwise_tukeyhsd(endog=df_anova['price'], groups=df_anova['cut'], alpha=0.05)
print('Resultados de Tukey HSD:')
print(tukey)

In [ ]:
fig = tukey.plot_simultaneous(figsize=(9, 5))
plt.title(
    'Tukey HSD — Intervalos de confianza simultáneos\n'
    '(grupos que no se solapan difieren significativamente)'
)
plt.xlabel('Precio ($)')
plt.tight_layout()
plt.show()

### Alternativa no paramétrica: Kruskal-Wallis

Cuando las condiciones de ANOVA no se cumplen, usamos **Kruskal-Wallis** (extensión de Mann-Whitney a ≥ 3 grupos):

- Trabaja con rangos
- No asume normalidad
- H₀: las distribuciones de todos los grupos son iguales

In [ ]:
H, p_kw = kruskal(*grupos_cut)
print(f'Kruskal-Wallis: H={H:.4f}, p={p_kw:.6f}')
print(f'{"✅ Diferencias significativas" if p_kw < 0.05 else "❌ Sin diferencias"}')
print()

print('Normalidad por grupo (Shapiro):')
for cut_val in orden_cut:
    grupo = diamonds.loc[diamonds['cut'] == cut_val, 'price'].values
    _, p_n = shapiro(grupo[:500])
    print(f'  {cut_val:<12} p={p_n:.5f} → {"✅" if p_n > 0.05 else "⚠️ no normal"}')

print()
print('→ El precio no es normal en ningún grupo → Kruskal-Wallis es más apropiado.')

In [ ]:
def comparar_grupos(df, col_num, col_grupo, alpha=0.05):
    """Compara una variable numérica entre múltiples grupos con diagnóstico automático."""
    datos = df[[col_num, col_grupo]].dropna()
    categorias = sorted(datos[col_grupo].unique())
    grupos = [datos.loc[datos[col_grupo] == c, col_num].values for c in categorias]

    print('=' * 65)
    print(f"COMPARACIÓN: '{col_num}' por '{col_grupo}'  ({len(categorias)} grupos)")
    print('=' * 65)

    desc = datos.groupby(col_grupo)[col_num].agg(['mean', 'std', 'count']).round(3)
    desc.columns = ['Media', 'Std', 'n']
    print(desc.to_string())
    print()

    normal_todos = all(
        shapiro(g[:500] if len(g) > 500 else g)[1] > alpha
        for g in grupos
    )
    _, p_lev = levene(*grupos)
    homo = p_lev > alpha
    print(f'  Normalidad: {"✅ todos normales" if normal_todos else "⚠️ alguno no normal"}')
    print(f'  Levene: p={p_lev:.4f} → {"✅ homocedástico" if homo else "⚠️ heterocedástico"}')
    print()

    if normal_todos and homo:
        stat, p = f_oneway(*grupos)
        print(f'  Test: ANOVA  →  F={stat:.4f}, p={p:.6f}')
    else:
        stat, p = kruskal(*grupos)
        print(f'  Test: Kruskal-Wallis  →  H={stat:.4f}, p={p:.6f}')

    conclusion = '✅ Diferencias significativas' if p < alpha else '❌ Sin diferencias'
    print(f'  Conclusión: {conclusion}')

    if p < alpha and normal_todos and homo:
        print()
        print('  Post-hoc Tukey HSD (pares con diferencia significativa):')
        tukey = pairwise_tukeyhsd(datos[col_num], datos[col_grupo], alpha=alpha)
        df_tukey = pd.DataFrame(data=tukey._results_table.data[1:],
                                 columns=tukey._results_table.data[0])
        sig = df_tukey[df_tukey['reject'] == True][['group1', 'group2', 'meandiff', 'p-adj']]
        print(sig.to_string(index=False) if len(sig) else '  Ningún par significativo.')

    print('=' * 65)
    print()

comparar_grupos(penguins, 'body_mass_g', 'species')

---
## 4.4 Información mutua (Mutual Information)

### ¿Qué mide?

La **información mutua** mide la **dependencia estadística** entre dos variables, incluyendo relaciones no lineales. A diferencia de la correlación de Pearson:

- Detecta relaciones **no lineales y no monótonas**
- Funciona para cualquier tipo de variable
- MI = 0 → independencia; MI > 0 → algún tipo de dependencia

$$MI(X, Y) = \sum_{x,y} p(x,y) \log\frac{p(x,y)}{p(x)p(y)}$$

In [ ]:
np.random.seed(0)
x = np.linspace(-3, 3, 500)
y_lineal = 0.8 * x + np.random.normal(0, 0.5, 500)
y_cuad   = x**2 + np.random.normal(0, 0.3, 500)
y_seno   = np.sin(2*x) + np.random.normal(0, 0.2, 500)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, y, titulo in zip(axes,
                          [y_lineal, y_cuad, y_seno],
                          ['Lineal', 'Cuadrática', 'Sinusoidal']):
    r, _ = pearsonr(x, y)
    mi = mutual_info_regression(x.reshape(-1, 1), y, random_state=0)[0]
    ax.scatter(x, y, s=8, alpha=0.5, color='steelblue')
    ax.set_title(f'{titulo}\nPearson r={r:.3f}  |  MI={mi:.3f}')
    ax.set_xlabel('x')

plt.suptitle('Pearson vs. Mutual Information: relaciones no lineales', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('→ Pearson no detecta la relación cuadrática ni la sinusoidal (r ≈ 0).')
print('  La MI sí las captura porque mide dependencia, no solo correlación lineal.')

In [ ]:
# MI para clasificación: relevancia vs 'survived'
df_mi = titanic[['age', 'fare', 'sibsp', 'parch', 'pclass', 'survived']].dropna().copy()
df_mi['pclass'] = df_mi['pclass'].astype(float)

X = df_mi.drop('survived', axis=1)
y = df_mi['survived']

mi_scores = mutual_info_classif(X, y, random_state=42)
mi_df = pd.DataFrame({'Variable': X.columns, 'MI': mi_scores}).sort_values('MI', ascending=False)

print("Mutual Information — variables vs 'survived':")
print(mi_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4))
colores = ['steelblue' if m > 0.01 else 'lightgray' for m in mi_df['MI']]
ax.barh(mi_df['Variable'], mi_df['MI'], color=colores, edgecolor='white')
ax.axvline(0.01, color='crimson', linestyle='--', linewidth=1.5, label='Umbral 0.01')
ax.set_xlabel('Mutual Information')
ax.set_title("MI con 'survived' — Titanic", fontsize=12)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# MI para regresión: relevancia vs precio en Diamonds
df_d = diamonds[['carat', 'depth', 'table', 'x', 'y', 'z', 'price']].dropna()
X_d = df_d.drop('price', axis=1)
y_d = df_d['price']

mi_reg = mutual_info_regression(X_d, y_d, random_state=42)
mi_reg_df = pd.DataFrame({'Variable': X_d.columns, 'MI': mi_reg}).sort_values('MI', ascending=False)

r_pearson = X_d.corrwith(y_d).abs().rename('|Pearson r|')
comparacion = mi_reg_df.set_index('Variable').join(r_pearson)
comparacion.columns = ['MI', '|Pearson r|']
comparacion = comparacion.sort_values('MI', ascending=False)

print("MI vs Pearson — Variables vs 'price' (Diamonds):")
print(comparacion.round(4))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].barh(comparacion.index, comparacion['MI'], color='steelblue', edgecolor='white')
axes[0].set_title('Mutual Information')
axes[0].set_xlabel('MI')

axes[1].barh(comparacion.index, comparacion['|Pearson r|'], color='coral', edgecolor='white')
axes[1].set_title('|Correlación de Pearson|')
axes[1].set_xlabel('|r|')

plt.suptitle("MI vs Pearson — Relevancia para 'price' (Diamonds)", fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4.5 Documentación de resultados y toma de decisiones para el modelado

La verificación estadística debe **documentarse** de forma que el equipo pueda entender qué se analizó, qué se encontró y qué decisiones se tomaron. Este es el objetivo del CE c del RA2.

### ¿Qué debe incluir un informe de verificación estadística?

1. **Descripción del dataset:** tamaño, variables, tipo
2. **Estadísticos descriptivos:** media, mediana, dispersión, sesgo
3. **Análisis de distribuciones:** normalidad, transformaciones aplicadas
4. **Análisis de relaciones:** correlaciones, tests de independencia
5. **Decisiones de diseño:** qué variables se incluyen o descartan y por qué

In [ ]:
def informe_estadistico_completo(df, col_objetivo, cols_num, cols_cat=None, alpha=0.05):
    """Informe estadístico completo para orientar la selección de variables."""
    from math import sqrt
    from scipy.stats import chi2_contingency, shapiro

    datos = df[cols_num + ([col_objetivo] if col_objetivo else [])].dropna()
    sep = '─' * 65

    print('=' * 65)
    print('  INFORME DE VERIFICACIÓN ESTADÍSTICA')
    print('=' * 65)

    print('\n1. DATASET')
    print(sep)
    print(f'   Filas (con variables completas): {len(datos)}')
    print(f'   Variables numéricas analizadas: {cols_num}')
    print(f'   Variable objetivo: {col_objetivo}')

    print('\n2. ESTADÍSTICOS DESCRIPTIVOS')
    print(sep)
    desc = datos[cols_num].describe().T
    desc['sesgo'] = datos[cols_num].skew().round(3)
    desc['alerta'] = desc['sesgo'].abs().apply(
        lambda s: '⚠️ transformar' if s > 1 else ('〰️ moderado' if s > 0.5 else '✅')
    )
    print(desc[['mean', 'std', '50%', 'sesgo', 'alerta']].rename(
        columns={'mean': 'Media', 'std': 'Std', '50%': 'Mediana',
                 'sesgo': 'Sesgo', 'alerta': 'Alerta'}).to_string())

    print('\n3. NORMALIDAD (Shapiro-Wilk)')
    print(sep)
    for col in cols_num:
        serie = datos[col]
        _, p_n = shapiro(serie[:500] if len(serie) > 500 else serie)
        estado = '✅ normal' if p_n > alpha else '⚠️ no normal'
        print(f'   {col:<28} p={p_n:.4f}  {estado}')

    print(f'\n4. RELEVANCIA RESPECTO A "{col_objetivo}"')
    print(sep)
    X_mi = datos[cols_num]
    y_mi = datos[col_objetivo]
    try:
        mi = mutual_info_classif(X_mi, y_mi, random_state=42)
    except Exception:
        mi = mutual_info_regression(X_mi, y_mi, random_state=42)
    for col, mi_val in sorted(zip(cols_num, mi), key=lambda x: -x[1]):
        r, p_r = pearsonr(datos[col], y_mi)
        rel = '✅ relevante' if mi_val > 0.01 or abs(r) > 0.1 else '❌ baja relevancia'
        print(f'   {col:<28} r={r:+.3f}  MI={mi_val:.4f}  {rel}')

    print('\n5. MULTICOLINEALIDAD (umbral |r| > 0.85)')
    print(sep)
    r_mat = datos[cols_num].corr()
    hay = False
    for i, c1 in enumerate(cols_num):
        for j, c2 in enumerate(cols_num):
            if j <= i: continue
            r = r_mat.loc[c1, c2]
            if abs(r) > 0.85:
                print(f'   ⚠️  {c1} vs {c2}: r={r:.3f}')
                hay = True
    if not hay:
        print('   ✅ No se detecta multicolinealidad severa.')

    if cols_cat:
        print(f'\n6. VARIABLES CATEGÓRICAS vs "{col_objetivo}"')
        print(sep)
        for col in cols_cat:
            tmp = df[[col, col_objetivo]].dropna()
            tabla = pd.crosstab(tmp[col], tmp[col_objetivo])
            chi2, p_chi, dof, _ = chi2_contingency(tabla)
            n = tabla.sum().sum()
            V = sqrt(chi2 / (n * (min(tabla.shape) - 1)))
            sig = '✅' if p_chi < alpha else '❌'
            print(f'   {col:<28} χ²={chi2:.2f}  p={p_chi:.4f}  V={V:.3f}  {sig}')

    print('\n' + '=' * 65)

# Aplicación al Titanic
cols_n = ['age', 'fare', 'sibsp', 'parch', 'pclass']
cols_c = ['sex', 'embarked', 'who']
df_rep = titanic.copy()
df_rep['pclass'] = df_rep['pclass'].astype(float)
df_rep['survived'] = df_rep['survived'].astype(float)

informe_estadistico_completo(df_rep, 'survived', cols_n, cols_c)

In [ ]:
# Aplicación al dataset Penguins
penguins_enc = penguins.copy()
penguins_enc['species_cod'] = penguins_enc['species'].map(
    {'Adelie': 0, 'Chinstrap': 1, 'Gentoo': 2}
)
cols_peng = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
informe_estadistico_completo(penguins_enc, 'species_cod', cols_peng)

### Guía de decisión: ¿qué variables incluir en el modelo?

In [ ]:
guia = """
┌─────────────────────────────────────────────────────────────────┐
│            GUÍA DE SELECCIÓN DE VARIABLES                       │
├─────────────────────────────────────────────────────────────────┤
│                                                                 │
│  1. ¿Tiene baja varianza (CV < 5%)?                             │
│     SÍ → DESCARTAR (no aporta información)                      │
│                                                                 │
│  2. ¿Tiene baja relevancia con el objetivo?                     │
│     (|r| < 0.1 Y MI < 0.01)                                     │
│     SÍ → candidata a DESCARTAR                                  │
│                                                                 │
│  3. ¿Está altamente correlacionada con otra variable?           │
│     (|r| > 0.85)                                                │
│     SÍ → CONSERVAR la de mayor relevancia o interpretabilidad   │
│                                                                 │
│  4. ¿Tiene sesgo importante (|sesgo| > 1)?                      │
│     SÍ → TRANSFORMAR antes de entrenar (log1p, sqrt, Box-Cox)   │
│                                                                 │
│  5. ¿Pasa todos los filtros anteriores?                         │
│     SÍ → INCLUIR en el modelo                                   │
│                                                                 │
│  Siempre documentar las decisiones con el test estadístico.     │
└─────────────────────────────────────────────────────────────────┘
"""
print(guia)

---
## 🔧 Ejercicios

### Ejercicio 1 — ANOVA en Penguins

Usa `comparar_grupos()` para analizar si `bill_length_mm` difiere entre las tres especies. Si es significativo, interpreta el resultado de Tukey: ¿qué pares difieren y cuáles no?

In [ ]:
# comparar_grupos(penguins, 'bill_length_mm', 'species')

# Visualiza con boxplot coloreado por especie
# sns.boxplot(data=penguins, x='species', y='bill_length_mm', palette='Set2')
# plt.show()

# ¿Qué par de especies es más difícil de distinguir por esta variable?

### Ejercicio 2 — Mutual Information vs Pearson

Para el dataset `diamonds`, calcula MI y correlación de Pearson de `carat`, `x`, `y`, `z`, `depth` y `table` respecto al precio. ¿Hay alguna variable donde MI y Pearson den señales distintas?

In [ ]:
# X_d = diamonds[['carat', 'x', 'y', 'z', 'depth', 'table']]
# y_d = diamonds['price']
# mi_scores = mutual_info_regression(X_d, y_d, random_state=42)
# r_scores = X_d.corrwith(y_d).abs()
# comparacion = pd.DataFrame({'MI': mi_scores, '|Pearson|': r_scores.values},
#                             index=X_d.columns).sort_values('MI', ascending=False)
# print(comparacion.round(4))

### Ejercicio 3 — Informe completo para Diamonds

Aplica `informe_estadistico_completo()` con `price` como objetivo y `['carat', 'depth', 'table', 'x', 'y', 'z']` como variables numéricas. Luego:
1. ¿Qué variables descartarías por multicolinealidad?
2. ¿Qué variables transformarías por sesgo?
3. ¿Cuál es el conjunto mínimo que retendrías? Justifica.

In [ ]:
# informe_estadistico_completo(
#     diamonds, 'price', ['carat', 'depth', 'table', 'x', 'y', 'z']
# )

# 1. Descartar por multicolinealidad: ...
# 2. Transformar por sesgo: ...
# 3. Conjunto mínimo recomendado: ...

---
## 📝 Resumen de la sesión

| Concepto | Clave |
|----------|-------|
| **ANOVA** | Compara medias de ≥ 3 grupos. F = varianza entre / varianza dentro. |
| **Kruskal-Wallis** | Alternativa no paramétrica a ANOVA. Usa rangos. |
| **Tukey HSD** | Post-hoc tras ANOVA significativo. Identifica pares que difieren. |
| **Mutual Information** | Mide dependencia general (lineal y no lineal). MI=0 → independencia. |
| **Informe estadístico** | Documenta: descriptivos, normalidad, relevancia, multicolinealidad. |
| **Selección de variables** | Descartar: baja varianza, baja relevancia. Fusionar: alta colinealidad. Transformar: sesgo. |

---

## 🎓 Resumen de la Unidad Didáctica 2

| Bloque | Contenidos | Técnicas clave |
|--------|-----------|----------------|
| **B1** | Estadística descriptiva | Media, mediana, std, IQR, percentiles, CV |
| **B2** | Distribuciones y normalidad | Normal/uniforme/binomial, sesgo, curtosis, Shapiro-Wilk, Q-Q, Box-Cox |
| **B3** | Correlación | Pearson, Spearman, Cramér's V, p-valor, IC, multicolinealidad |
| **B4** | Tests de independencia | Chi-cuadrado, t-test, Mann-Whitney, ANOVA, Tukey, Mutual Information |

> El hilo conductor de toda la unidad: **¿qué nos dicen estos datos sobre el fenómeno que queremos modelar, y qué variables debemos incluir o descartar?**